# Task 4 — Open-Ended Business Analytics Solution

Proposed solution: business sales and customer analytics. Add the official Task 4 dataset to `data/raw/` with `sales`, `business` or `customer` in the filename. Confirm that this project matches the official Task 4 brief before submission.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
DATA = ROOT / "data" / "raw"
files = list(DATA.glob("*.csv")) + list(DATA.glob("*.xlsx")) + list(DATA.glob("*.json"))
print("Data files found:", [f.name for f in files])
print("Put the correct task dataset in data/raw/ before interpreting results.")

def load_data(keywords):
    matches = [f for f in files if any(k in f.name.lower() for k in keywords)]
    if not matches:
        print("DATASET MISSING for:", keywords)
        return None
    f = matches[0]
    print("Loading:", f.name)
    if f.suffix.lower() == ".csv":
        return pd.read_csv(f)
    if f.suffix.lower() == ".xlsx":
        return pd.read_excel(f)
    return pd.read_json(f)

def overview(df):
    if df is None:
        return
    print("Shape:", df.shape)
    display(df.head())
    display(df.isna().sum().sort_values(ascending=False).head(20).to_frame("missing"))
    display(df.describe(include="all").T)

df = load_data(['sales', 'business', 'customer'])
overview(df)

Data files found: []
Put the correct task dataset in data/raw/ before interpreting results.
DATASET MISSING for: ['sales', 'business', 'customer']


## KPI and trend exploration

In [2]:
if df is not None:
    numeric = df.select_dtypes(include="number")
    print("Numeric KPI candidates:", list(numeric.columns))
    for col in numeric.columns[:8]:
        print(f"{col}: sum={numeric[col].sum():,.2f}; mean={numeric[col].mean():,.2f}")
    date_cols = [c for c in df.columns if any(
        k in c.lower() for k in ["date", "order_time", "timestamp"]
    )]
    if date_cols:
        print("Date candidates:", date_cols)
        parsed = pd.to_datetime(df[date_cols[0]], errors="coerce")
        if parsed.notna().any() and len(numeric.columns):
            tmp = df.assign(_date=parsed).dropna(subset=["_date"])
            metric = numeric.columns[0]
            trend = tmp.groupby(tmp["_date"].dt.to_period("M"))[metric].sum()
            display(trend.to_frame(metric))
            trend.plot(title=f"Monthly total: {metric}", figsize=(10, 4))
            plt.tight_layout()
            plt.show()
    print("Validate currency, units, duplicate orders, returns and missing values before conclusions.")
else:
    print("Dataset missing. Add the official data before writing findings.")

Dataset missing. Add the official data before writing findings.


## Business recommendations

Use measured KPIs to identify opportunities, quantify limitations and recommend a testable action. Include the dataset source, assumptions, evaluation method and reproducible steps.